In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import torch

from helmholtz_eos_net.test import load_model_from_folder


msize=12
mwidth = 2
fsize=16
alpha=0.3
lsize = 2
framewidth = 3
markeredgewidth=3
markeredgewidth_no=10

print('Imports successful!')

In [ ]:
# the base model in the paper
model_folder = "ray_results/d29fd_00000/"

# the water model in the paper
model_folder = "trained_models/transport_joint_water_res_08c6/" 


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, config = load_model_from_folder(model_folder, device)
model.eval()

In [ ]:
# Single VLE reference for the optimization and all three figures: the NIST
# WebBook saturation table (IAPWS-95). Carries p_sat, rho_liq, rho_vap, dHvap
# and the SAFT parameters, so it also feeds branch_input below.
vle_df   = pd.read_csv('water_nist/water_nist_vle.csv')

# Homogeneous p-rho-T backdrop — the actual residual training split.
state_df = pd.read_csv('water_data/data_res_full_train.csv')


# Temperature grid — change T_step to adjust resolution
T_step = 5.0
#T_step = 3.0
#T_step = 1.0
T_min  = vle_df['temperature_K'].min() + 5 #T_step
T_max  = vle_df['temperature_K'].max() - 20 # T_step
temps  = np.arange(T_min, T_max, T_step)
print(f'Evaluating VLE at {len(temps)} temperatures ({T_min:.1f}–{T_max:.1f} K, step={T_step} K)')

In [ ]:
from feos.pcsaft import PcSaftParameters, PcSaftRecord, Identifier, PureRecord
from feos.eos import EquationOfState, State, PhaseDiagram
from feos.eos import Contributions
from si_units import * 

eos_csv = "SI_pcp-saft_parameters.csv"

eos_keys = ["m","sigma","epsilon_k","kappa_ab","epsilon_k_ab","mu"]
embedding_keys = ["molarweight","m","sigma","epsilon_k","kappa_ab","epsilon_k_ab","mu"]

embedding_paras = pd.read_csv( eos_csv, sep="\t")
embedding_paras = embedding_paras.fillna( 0.0 )

print( np.where( embedding_paras.common_name == "water") )
sub = embedding_paras.iloc[156]
name = "water"


M = sub["molarweight"]*(GRAM/MOL)
m = sub["m"]

identifier = Identifier( cas="000-00-0", name="dummy" )
saftrec = PcSaftRecord( m=sub["m"], sigma=sub["sigma"], epsilon_k=sub["epsilon_k"],
                      kappa_ab=sub["kappa_ab"],epsilon_k_ab=sub["epsilon_k_ab"],mu=sub["mu"],
                       na=sub["na"], nb=sub["nb"] )

pr = PureRecord(identifier=identifier,model_record=saftrec,molarweight=sub["molarweight"])

saft_paras = PcSaftParameters.from_records( [pr], np.array([[0.0]] ) )

eos = EquationOfState.pcsaft(saft_paras)

saft_vle = PhaseDiagram.pure(eos, min_temperature=273.2*KELVIN , npoints=100)

temperature_saft = saft_vle.vapor.temperature / KELVIN
pressure_saft = saft_vle.vapor.pressure / PASCAL
rho_vap_saft = saft_vle.vapor.density / MOL * METER**3
rho_liq_saft = saft_vle.liquid.density / MOL * METER**3
h_vap = saft_vle.vapor.molar_enthalpy(Contributions.Residual)
h_liq = saft_vle.liquid.molar_enthalpy(Contributions.Residual)
dh_vap_saft = (h_vap - h_liq) /JOULE *MOL


dh_vap_saft


In [ ]:
# Interpolate initial guesses from the NIST reference.
# f_psat / f_gas / f_liq are reused below as the parameter-optimization targets.
vle_sorted = vle_df.sort_values('temperature_K')
f_gas = interp1d(vle_sorted['temperature_K'], vle_sorted['molar_density_vapor_mol_m3'],  kind='linear', fill_value='extrapolate')
f_liq = interp1d(vle_sorted['temperature_K'], vle_sorted['molar_density_liquid_mol_m3'], kind='linear', fill_value='extrapolate')
f_psat = interp1d(vle_sorted.groupby('temperature_K')['pressure_sat_Pa'].first().index,
                  vle_sorted.groupby('temperature_K')['pressure_sat_Pa'].first().values,
                  kind='linear', fill_value='extrapolate')

rho_gas_init = torch.tensor( 1* f_gas(temps)[:, None], dtype=torch.float32)
rho_liq_init = torch.tensor( 1* f_liq(temps)[:, None], dtype=torch.float32)
T_tensor     = torch.tensor(temps[:, None], dtype=torch.float32)

# Substance parameters (same for all water rows)
branch_input = torch.tensor(
    np.tile(vle_df[model.substance_params.parameter_columns].iloc[0].values, (len(temps), 1)),
    dtype=torch.float32
)

print(f'T_tensor:     {T_tensor.shape}')
print(f'rho_gas_init: {rho_gas_init.shape}')
print(f'rho_liq_init: {rho_liq_init.shape}')
print(f'branch_input: {branch_input.shape}')

In [ ]:
branch_input

In [ ]:
if False:
    branch_input[:,0] = branch_input[:,0] + 0.01
    branch_input[:,1] = branch_input[:,1] + 0.01
    branch_input[:,2] = branch_input[:,2] + 0.01

    branch_input[:,0] = branch_input[:,0] - 0.03
    branch_input[:,1] = branch_input[:,1] - 0.02
    branch_input[:,2] = branch_input[:,2] - 0.01

## Parameter optimization

The substance parameters in `branch_input` (columns given by `model.substance_params.parameter_columns`) were hand-picked above and never fitted. Two columns — `is_associating` and `has_dipole` — are boolean switches and are kept fixed; the other 6 continuous columns (`m_norm`, `sigma_norm`, `epsilon_k_norm`, `kappa_ab_val`, `epsilon_k_ab_val`, `mu_val`) are optimized below against the NIST reference curves (`f_psat`, `f_gas`, `f_liq`) that are already interpolated above.

> **Note on what the reported errors mean.** Since the switch to a single NIST reference, the optimization targets and the reference plotted in the three figures below are the *same* curve. The MAPEs reported there are therefore in-sample with respect to these 5 fitted parameters (the model weights themselves remain frozen and were trained without any VLE data). The unfitted counterpart is `test_model_vle_water_fast.ipynb`, which evaluates the hand-picked parameters against NIST directly.

**Why `scipy.optimize` + backprop, not a hand-rolled training loop:**

`HEOSNet.forward_normalized` computes its Helmholtz-energy derivatives via `torch.autograd.grad(..., create_graph=True)`, so the whole model — including its dependence on the substance-parameter vector `p_input` — is differentiable. `find_vle_flash_batch` (and `find_vle_curve`, which calls it) `.detach()`es the *density* iterate at every Newton step, by design, for solver stability — but the final `p_sat`/`mu_sat` come from one more differentiable `forward()` call, so `loss.backward()` still yields an exact analytic gradient of the loss w.r.t. the 6 free parameters (it just omits the implicit `∂ρ*/∂p_input` term — a standard, effective approximation for this class of fixed-point solver).

Since an exact gradient is available for free via autograd, there's no reason to fall back to scipy's finite-difference gradient (which would cost ~13 extra full VLE-batch evaluations per iteration, each running a 40-step Newton solve per temperature). Instead we hand that analytic gradient to `scipy.optimize.minimize(method="L-BFGS-B")` — a bounded quasi-Newton optimizer that is more robust than a manually-tuned Adam/SGD loop for a small (6-parameter), deterministic, well-scaled problem, with built-in box constraints and line search.

In [ ]:
import time
from scipy.optimize import minimize

# Freeze all model weights — only the substance parameters are being optimized
for p in model.parameters():
    p.requires_grad_(False)

param_columns = model.substance_params.parameter_columns
fixed_names = ['is_associating', 'has_dipole', 'mu_val']
fixed_idx = [param_columns.index(name) for name in fixed_names]
free_idx  = [i for i in range(len(param_columns)) if i not in fixed_idx]

print(f'Parameter columns: {param_columns}')
print(f'Fixed (boolean):   {[param_columns[i] for i in fixed_idx]}')
print(f'Free (continuous): {[param_columns[i] for i in free_idx]}')

#branch_input_orig = branch_input.clone()
branch_input_orig =  np.array([0.06113127, 0.09625337, 0.3628961, 1 , 0.89342838, 0.41451652, 0,0])
branch_input_orig = torch.tensor(np.atleast_2d(branch_input_orig), dtype=torch.float32)

# Reference values at the `temps` evaluation grid (reuses the interpolators built above)
p_sat_ref_opt   = f_psat(temps)
rho_gas_ref_opt = f_gas(temps)
rho_liq_ref_opt = f_liq(temps)


In [ ]:
n_free = len(free_idx)

def _build_branch_tensor(x_free):
    """Scatter the free-parameter vector into a fresh (n_temps, 8) branch tensor."""
    x_full = branch_input_orig[0].clone()
    x_full[free_idx] = torch.as_tensor(x_free, dtype=torch.float32)
    return x_full.unsqueeze(0).repeat(len(temps), 1)

def objective_and_grad_ini(z, x0):
    x = z * x0

    branch_tensor = _build_branch_tensor(x)
    #branch_tensor.requires_grad_(True)

    with torch.enable_grad():
        p_sat_pred, _, rho_gas_pred, rho_liq_pred, converged = model.find_vle_flash_batch(
            branch_tensor, T_tensor, rho_gas_init, rho_liq_init,
            n_steps=40, tol=1e-6, alpha=0.5,
        )

    p_sat_pred   = p_sat_pred.squeeze(-1)
    rho_gas_pred = rho_gas_pred.squeeze(-1)
    rho_liq_pred = rho_liq_pred.squeeze(-1)
    converged    = converged.squeeze(-1)

    p_sat_ref_t   = torch.as_tensor(p_sat_ref_opt,   dtype=torch.float32)
    rho_gas_ref_t = torch.as_tensor(rho_gas_ref_opt, dtype=torch.float32)
    rho_liq_ref_t = torch.as_tensor(rho_liq_ref_opt, dtype=torch.float32)

    #mask = (converged
    #        & torch.isfinite(p_sat_pred)
    #        & (p_sat_ref_t > 0) & (rho_gas_ref_t > 0) & (rho_liq_ref_t > 0))

    mask = (p_sat_ref_t > 0) & (rho_gas_ref_t > 0) & (rho_liq_ref_t > 0)   # constant across x

    LOSS_SCALE = 1e10 

    if mask.sum() == 0:
        # Trial step converged nowhere — treat as strictly worse, don't hand scipy a NaN.
        return 1e3 * LOSS_SCALE #, np.zeros(n_free, dtype=np.float64)

    def rel_mse(pred, ref):
        return (((pred[mask] - ref[mask]) / ref[mask]) ** 2).mean()

    def rel_mse_log(pred, ref):
        return ((( torch.log(pred[mask]) - torch.log(ref[mask])) / torch.log(ref[mask])) ** 2).mean()        

    #loss = rel_mse(p_sat_pred, p_sat_ref_t) + rel_mse(rho_liq_pred, rho_liq_ref_t) + rel_mse(rho_gas_pred, rho_gas_ref_t)
    loss = rel_mse_log(p_sat_pred, p_sat_ref_t) + 2*rel_mse(rho_liq_pred, rho_liq_ref_t) # + rel_mse_log(rho_gas_pred, rho_gas_ref_t)
    #loss = rel_mse_log(p_sat_pred, p_sat_ref_t) + rel_mse(rho_liq_pred, rho_liq_ref_t)

    
    loss = LOSS_SCALE * loss

    #loss.backward()
    #grad = branch_tensor.grad[:, free_idx].sum(dim=0).numpy().astype(np.float64)
    #grad = np.nan_to_num(grad, nan=0.0, posinf=1e3, neginf=-1e3)
    loss_val = float(loss.item())
    if not np.isfinite(loss_val):
        loss_val = 1e3 * LOSS_SCALE
    return loss_val #, grad

In [ ]:
"""

x0 = branch_input_orig[0, free_idx].numpy().astype(np.float64)
objective_and_grad = lambda z: objective_and_grad_ini(z, x0)

z0       = np.ones(n_free)
bounds_z = [(0.0, 1.0 / x0[i]) for i in range(n_free)]

jac=False
if jac:
    loss_before, _ = objective_and_grad(z0)
else:
    loss_before = objective_and_grad(z0)

t0 = time.time()
res = minimize(objective_and_grad, z0, jac=jac, method='L-BFGS-B', bounds=bounds_z,
                options={'maxiter': 100, 'maxls': 50, 'eps': 1e-2, 'ftol': 1e-12, 'gtol': 1e-12}
)
#res = minimize(objective_and_grad, x0)


res.x = res.x * x0      # back to physical units for the comparison frame and downstream cells

print(f'Optimization finished in {time.time()-t0:.1f}s, {res.nit} iterations, converged={res.success}')
print(f'Status: {res.status} — {res.message}')
print(f'Loss before: {loss_before:.6f}')
print(f'Loss after:  {res.fun:.6f}')

param_names_free = [param_columns[i] for i in free_idx]
comparison = pd.DataFrame({
    'parameter': param_names_free,
    'before':    x0,
    'after':     res.x,
})
res0 = res
comparison


"""

In [ ]:
"""
rng = np.random.default_rng(0)
n_restarts =0
jitter_scale = 0.05  # relative perturbation of the initial guess (5% of each bound's range)

lo = np.array([b[0] for b in bounds_z])
hi = np.array([b[1] for b in bounds_z])

best_res = None
for i in range(n_restarts):
    z_start = z0.copy() if i == 0 else np.clip(
        z0 + rng.normal(scale=jitter_scale, size=n_free) * (hi - lo), lo, hi
    )
    print(z_start)
    res_i = minimize(objective_and_grad, z_start, jac=jac, method='L-BFGS-B', bounds=bounds_z,
                      options={'maxiter': 200, 'maxls': 50, 'ftol': 1e-10, 'gtol': 1e-10, 'eps': 1e-4})
    print(f'restart {i}: loss={res_i.fun:.6f}  nit={res_i.nit}  status={res_i.status}')
    if best_res is None or res_i.fun < best_res.fun:
        best_res = res_i

try:
    res = best_res
    print(f'\nBest loss: {res.fun:.6f}  (loss before: {loss_before:.6f})')
except:
    res = res0

comparison['after'] = res.x
comparison
"""

In [ ]:
"""
parameter	before	after
0	m_norm	0.061349	0.061408
1	sigma_norm	0.096431	0.096662
2	epsilon_k_norm	0.361431	0.357812
3	kappa_ab_val	0.897266	0.884000
4	epsilon_k_ab_val	0.415646	0.415975

array([0.06140821, 0.09666178, 0.35781246, 0.88400002, 0.41597518])


BEST:

array([0.06113127, 0.09625337, 0.3628961 , 0.89342838, 0.41451652])


"""

In [ ]:
#res = res.x
res = np.array([0.06113127, 0.09625337, 0.3628961 , 0.89342838, 0.41451652])

In [ ]:
# Reassign `branch_input` so all downstream cells (VLE curve, plots, MAPE summary)
# transparently use the optimized substance parameters.
branch_input = _build_branch_tensor(res).detach()
print('branch_input updated with optimized parameters:')
branch_input


# Run batch VLE flash
with torch.enable_grad():
    p_sat_pred, mu_sat_pred, rho_gas_vle, rho_liq_vle, converged = model.find_vle_flash_batch(
        branch_input, T_tensor, rho_gas_init, rho_liq_init,
        n_steps=40, tol=1e-6, alpha=0.5,
        # determine_rho_liq=0*torch.ones(T_tensor.shape[0], 1)  # freeze liquid at rho_liq_init
    )

rho_gas_vle = rho_gas_vle.detach().numpy().squeeze()
rho_liq_vle = rho_liq_vle.detach().numpy().squeeze()
converged   = converged.numpy()
p_sat_pred  = p_sat_pred.detach().numpy().squeeze()

temperature = T_out.numpy()

print(f'Converged: {converged.sum()}/{len(converged)}')

In [ ]:
# ── Stepped VLE curve: predictor-corrector from T_max downward ──────────────
p_water = branch_input[0:1]       # (1, 8) — single substance for all steps
T_1d    = T_tensor.squeeze()      # (n_T,) — any order, sorted internally

# Initial guess at the highest temperature in temps (NIST reference). Only these
# two scalars are consumed — find_vle_curve marches down from here.
rho_l0 = float(rho_liq_init[-1])
rho_g0 = float(rho_gas_init[-1])

with torch.enable_grad():
    if False:
        p_sat_out, mu_sat_pred, rho_gas_out, rho_liq_out, converged_out = model.find_vle_flash_batch(
            branch_input, T_tensor, rho_gas_init, rho_liq_init,
            n_steps=40, tol=1e-6, alpha=0.5,
            # determine_rho_liq=0*torch.ones(T_tensor.shape[0], 1)  # freeze liquid at rho_liq_init
        )        
        T_out = T_tensor
    else:
        T_out, p_sat_out, p_sat_cc_out, rho_liq_out, rho_gas_out, dHvap_out, converged_out = \
            model.find_vle_curve(p_water, T_1d, rho_l0, rho_g0, n_steps=40)
            

temperature = T_out.numpy()
rho_gas_vle = rho_gas_out.detach().numpy()
rho_liq_vle = rho_liq_out.detach().numpy()
p_sat_pred  = p_sat_out.detach().numpy()
p_sat_cc    = p_sat_cc_out.detach().numpy()
delta_H_vap = dHvap_out.detach().numpy()
converged   = converged_out.detach().numpy()

print(f'Converged: {converged.sum()}/{len(converged)}')


In [ ]:
#temperature = T_tensor.detach().numpy().squeeze()
#temperature = T_out.numpy()

plt.plot( state_df.molar_density_mol_m3, state_df.temperature_K, ".y" )

plt.plot( vle_df.molar_density_liquid_mol_m3, vle_df.temperature_K, "-g" )


plt.plot(rho_gas_vle, temperature, ".-m" )
plt.plot(rho_liq_vle, temperature, ".-m" )


In [ ]:

plt.plot( state_df.molar_density_mol_m3, state_df.temperature_K, ".y" )

plt.plot(rho_gas_vle, temperature, ".-m" )
plt.plot(rho_liq_vle, temperature, ".-m" )

In [ ]:
plt.plot( vle_df.temperature_K, vle_df.pressure_sat_Pa, ".y")

plt.plot( temperature, p_sat_pred, ".-m" )


In [ ]:


plt.plot( vle_df.temperature_K, np.log(vle_df.pressure_sat_Pa) , ".y")

plt.plot( temperature, np.log(p_sat_cc), ".-g" )

plt.plot( temperature, np.log(p_sat_pred), ".m" )

p = converged.squeeze() == False
plt.plot( temperature[p], np.log(p_sat_pred[p]), "xm" )


In [ ]:
temperature_grid, _ = np.meshgrid(temperature, temperature)
pressure_grid, _ = np.meshgrid(p_sat_pred, p_sat_pred)
delta_H_vap_grid, _ = np.meshgrid(delta_H_vap, delta_H_vap)

temperature_diag = np.diag(temperature_grid)[:, np.newaxis].repeat(temperature_grid.shape[0], axis=1)

t_diff = temperature_grid - temperature_diag
t_diff = np.abs(t_diff)
#weights = 1 - t_diff / t_diff.sum(axis=1, keepdims=True)
#weights[t_diff > 100] = 0
weights = 1/(1 + np.exp(t_diff/40))
weights = weights / weights.sum(axis=1, keepdims=True)

p_sat_ccc = pressure_grid * np.exp(delta_H_vap_grid / 8.314 * (1.0 / temperature_grid - 1.0 / temperature_diag))

p_sat_ccc = p_sat_ccc * weights
p_sat_ccc = p_sat_ccc.sum(axis=1)

plt.plot( temperature_saft, np.log(pressure_saft), "k--", label="PCP-SAFT")
plt.plot( vle_df.temperature_K, np.log(vle_df.pressure_sat_Pa) ,  ".", color="limegreen",markersize=msize, label="NIST / IAPWS-95")

#plt.plot( temperature, np.log(p_sat_ccc), ".-g" )

f_p_vap = interp1d( vle_df.temperature_K, np.log(vle_df.pressure_sat_Pa), fill_value="extrapolate" )

ref = np.exp(f_p_vap(temperature))
error = np.mean( np.abs( p_sat_pred -  ref) / ref )

plt.plot( temperature, np.log(p_sat_pred), "m.-",markersize=msize , label=f"water ({error*100:.1f}%)" )

ref = np.exp(f_p_vap(temperature_saft))
pcs_error = np.mean( np.abs( pressure_saft -  ref) / ref )

plt.plot( temperature_saft, np.log(pressure_saft), "k--")

plt.xlabel(r"temperature / K",fontsize=fsize)
plt.ylabel(r"$ln(p ~/~ Pas)$",fontsize=fsize)

plt.legend(fontsize=fsize, frameon=False)


plt.xticks(fontsize=fsize)  
plt.yticks(fontsize=fsize)  


plt.savefig(model_folder+f"VLE_{name}_pressure_optimized.png",bbox_inches='tight')
plt.savefig(model_folder+f"VLE_{name}_pressure_optimized.pdf",bbox_inches='tight')

print(f"p_sat  model {error*100:.2f}%   PC-SAFT {pcs_error*100:.2f}%")
error, pcs_error

In [ ]:
plt.plot( temperature_saft, dh_vap_saft/1000, "k--", label="PCP-SAFT")

plt.plot( vle_df.temperature_K, vle_df.molar_vaporization_enthalpy_J_mol/1000, ".", color="limegreen",markersize=msize, label="NIST / IAPWS-95" )

f_dh_vap = interp1d( vle_df.temperature_K, vle_df.molar_vaporization_enthalpy_J_mol)

ref = f_dh_vap(temperature)
error = np.mean( np.abs( delta_H_vap -  ref) / ref )


plt.plot( temperature, delta_H_vap/1000, "m.-",markersize=msize , label=f"water ({error*100:.2f}%)" )
# plt.plot( temperature[p], delta_H_vap[p]/1000, "xm" )

plt.plot( temperature_saft, dh_vap_saft/1000, "k--")

plt.xlabel(r"temperature / K",fontsize=fsize)
plt.ylabel(r"$\Delta h_{\text{vap}}~/~kJ/mol$",fontsize=fsize)

plt.legend(fontsize=fsize, frameon=False)

plt.xticks(fontsize=fsize)  
plt.yticks(fontsize=fsize)  


plt.savefig(model_folder+f"VLE_{name}_enthalpy_optimized.png",bbox_inches='tight')
plt.savefig(model_folder+f"VLE_{name}_enthalpy_optimized.pdf",bbox_inches='tight')

print(f"dHvap  model {error*100:.2f}%")

In [ ]:
def water_liquid_density(T_k):
    """
    Calculates the liquid density of pure water using DIPPR Equation 105.
    
    Parameters:
    -----------
    T_k : float
        Temperature in Kelvin (K). Range: 273.15 K to 647.13 K.
    method : str
        'piecewise' for high-accuracy temperature envelopes.
        'standard' for the single full-range regression.
        
    Returns:
    --------
    dict
        Molar density (kmol/m3) and Mass density (kg/m3).
    """
    # Molecular weight of water (kg/kmol)
    # MW = 18.01528  
    

    A, B, C, D = 5.459, 0.30542, 647.13, 0.081


    # DIPPR Equation 105 (Modified Rackett)
    # rho = A / (B ** (1 + (1 - T/Tc)**D))
    tau = 1.0 - (T_k / C)
    exponent = 1.0 + (tau ** D)
    molar_density = A / (B ** exponent)
    
    # Convert kmol/m3 to kg/m3
    #mass_density = molar_density * MW
    return molar_density

# very bad    
dcc = water_liquid_density(temperature)    
dcc = np.array(dcc)

In [ ]:

plt.plot( rho_liq_saft/1000, temperature_saft, "k--", label="PCP-SAFT")
#plt.plot( state_df.molar_density_mol_m3/1000, state_df.temperature_K, ".", color="limegreen", label="experimental train" )

plt.plot(vle_df.molar_density_vapor_mol_m3/1000, vle_df.temperature_K, ".",markersize=msize, color="limegreen" )
plt.plot(vle_df.molar_density_liquid_mol_m3/1000, vle_df.temperature_K, ".",markersize=msize, color="limegreen", label="NIST / IAPWS-95" )

f_rho_vap = interp1d( vle_df.temperature_K, vle_df.molar_density_vapor_mol_m3)
f_rho_liq = interp1d( vle_df.temperature_K, vle_df.molar_density_liquid_mol_m3)

ref = f_rho_vap(temperature)
error_vap = np.mean( np.abs( rho_gas_vle -  ref) / ref )

ref = f_rho_liq(temperature)
error_liq = np.mean( np.abs( rho_liq_vle -  ref) / ref )

plt.plot(rho_gas_vle/1000, temperature, ".-m",markersize=msize )
plt.plot(rho_liq_vle/1000, temperature, ".-m",markersize=msize , label=f"water (L {error_liq*100:.2f}%, V {error_vap*100:.2f}%)" )

plt.plot( rho_vap_saft/1000, temperature_saft, "k--")
plt.plot( rho_liq_saft/1000, temperature_saft, "k--")


plt.xlabel(r"$\rho~/~kmol/m^3$",fontsize=fsize)
plt.ylabel(r"temperature / K",fontsize=fsize)

plt.legend(fontsize=fsize, frameon=False)

plt.xticks(fontsize=fsize)  
plt.yticks(fontsize=fsize)  

plt.savefig(model_folder+f"VLE_{name}_density_optimized.png",bbox_inches='tight')
plt.savefig(model_folder+f"VLE_{name}_density_optimized.pdf",bbox_inches='tight')

error_liq, error_vap

In [ ]:
x = rho_gas_vle[ temperature < 350 ]
y = temperature[ temperature < 350 ]

plt.plot(x, y, ".-m",markersize=msize )

mm = np.max(x)

mm, np.log(mm), np.log(320)

In [ ]:
plt.plot( temperature, rho_gas_vle, "m." )
plt.plot( temperature, np.ones(temperature.shape)*np.exp(3), "k-" )

In [ ]:
# temperature, delta_H_vap, rho_gas_vle

dvvap = 1/rho_gas_vle - 1/rho_liq_vle

dpdt = delta_H_vap/dvvap

plt.plot(  np.log( np.sort(dpdt) ) )

In [ ]:

dvvap = 1/rho_gas_vle - 1/rho_liq_vle

dpdt = delta_H_vap/dvvap/p_sat_pred

plt.plot(  np.log( dpdt ) )